# Node specific parameters with torch training plan

The goal of this notebook is to test `tag_parameters` feature with torch training plan.

This example uses MNIST dataset. Please check `README.md` file in `notebooks` directory for the instructions to load MNIST dataset and configure nodes.

# Test 1: simple experiment with breakpoints

## Define an experiment model and parameters

Declare a torch training plan MyTrainingPlan class to send for training on the node

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torchvision import  transforms

from fedbiomed.common.training_plans import TorchTrainingPlan
from fedbiomed.common.datamanager import DataManager
from fedbiomed.common.optimizers.optimizer import Optimizer
from fedbiomed.common.optimizers.declearn import ScaffoldClientModule
from fedbiomed.common.dataset import MnistDataset


# Here we define the training plan to be used.
# You can use any class name (here 'MyTrainingPlan')
class MyTrainingPlan(TorchTrainingPlan):

    def init_dependencies(self):
        return ["from torchvision import transforms",
                "from fedbiomed.common.dataset import MnistDataset",
                "from fedbiomed.common.optimizers.optimizer import Optimizer",
                "from fedbiomed.common.optimizers.declearn import ScaffoldClientModule",
                "from fedbiomed.common.optimizers.declearn import AdamModule",
                "from fedbiomed.common.optimizers.declearn import RidgeRegularizer",
                "from torch.optim import Adam"]
        
    class Net(nn.Module):
        def __init__(self, model_args):
            super().__init__()
            self.conv1 = nn.Conv2d(1, 32, 3, 1)
            self.conv2 = nn.Conv2d(32, 64, 3, 1)
            self.dropout1 = nn.Dropout(0.25)
            self.dropout2 = nn.Dropout(0.5)
            self.fc1 = nn.Linear(9216, 128)
            self.fc2 = nn.Linear(128, 10)

        def forward(self, x):
            x = self.conv1(x)
            x = F.relu(x)
            x = self.conv2(x)
            x = F.relu(x)
            x = F.max_pool2d(x, 2)
            x = self.dropout1(x)
            x = torch.flatten(x, 1)
            x = self.fc1(x)
            x = F.relu(x)
            x = self.dropout2(x)
            x = self.fc2(x)
            output = F.log_softmax(x, dim=1)
            return output

    def init_model(self, model_args):
        model = self.Net(model_args = model_args)
        return model

    def init_optimizer(self, optimizer_args):
        return Adam(self.model().parameters(), lr = optimizer_args["lr"])

    def training_data(self):
        transform = transforms.Normalize((0.1307,), (0.3081,))
        loader_arguments = { 'shuffle': True}
        
        dataset = MnistDataset(transform=transform)
        return DataManager(dataset, **loader_arguments)

    def training_step(self, data, target):
        output = self.model().forward(data)
        loss   = torch.nn.functional.nll_loss(output, target)
        return loss

    def tag_parameters(self, name):
        if name == 'conv1.weight':
            return {'local', 'persistent'}
        if name == 'conv2.bias':
            return {'persistent'}
        return set()
            

### Model arguments and training arguments

In [2]:
model_args = {}

training_args = {
    'loader_args': { 'batch_size': 5, }, 
    'optimizer_args': {
        "lr" : 1e-3
    },
    'num_updates': 20,
    'dry_run': False
}

### Create and run the experiment

In [3]:
from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.aggregators.fedavg import FedAverage
from fedbiomed.common.optimizers.optimizer import Optimizer
from fedbiomed.common.optimizers.declearn import ScaffoldServerModule
from fedbiomed.common.optimizers.declearn import YogiModule as FedYogi

tags =  ['#MNIST', '#dataset']
rounds = 15

exp = Experiment(tags=tags,
                 model_args=model_args,
                 training_plan_class=MyTrainingPlan,
                 training_args=training_args,
                 round_limit=rounds,
                 aggregator=FedAverage(),
                 tensorboard=True,
                 save_breakpoints=True,
                 node_selection_strategy=None)

2026-04-15 17:12:26,563 fedbiomed INFO - Syslog configuration is disabled or not found in configuration. Disabling syslog logging.

2026-04-15 17:12:26,605 fedbiomed INFO - Starting researcher service...

2026-04-15 17:12:26,606 fedbiomed INFO - Waiting 3s for nodes to connect...

2026-04-15 17:12:29,612 fedbiomed INFO - Updating training data. This action will update FederatedDataset, and the nodes that will participate to the experiment.

2026-04-15 17:12:29,686 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_5eaa634a-8589-4251-9779-f4435535cb17

2026-04-15 17:12:29,687 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_523186bc-f4af-4044-a946-60003cdef368

/user/lchambon/home/miniconda3/envs/fedbiomed-dev/lib/python3.10/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


In [4]:
exp.run()

2026-04-15 17:12:29,732 fedbiomed INFO - Sampled nodes in round 0 ['NODE_5eaa634a-8589-4251-9779-f4435535cb17', 'NODE_523186bc-f4af-4044-a946-60003cdef368']

2026-04-15 17:12:29,745 fedbiomed INFO - Sending request 
					 To: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-04-15 17:12:29,747 fedbiomed INFO - Sending request 
					 To: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-04-15 17:12:29,908 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Node Name: Default Node Alias 
					 Round 1 | Iteration: 1/20 (5%) | Samples: 5/100
 					 Loss: 2.330683 
					 ---------

2026-04-15 17:12:30,537 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Node Name: Default Node Alias 
					 Round 1 | Iteration: 1/20 (5%) | Samples: 5/100
 					 Loss: 2.316147 
					 ---------

...

2026-04-15 17:14:18,467 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Node Name: Default Node Alias 
					 Round 15 | Iteration: 20/20 (100%) | Samples: 100/100
 					 Loss: 0.670227 
					 ---------

2026-04-15 17:14:18,526 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Node Name: Default Node Alias 
					 Round 15 | Iteration: 20/20 (100%) | Samples: 100/100
 					 Loss: 1.569737 
					 ---------

2026-04-15 17:14:18,922 fedbiomed INFO - Nodes that successfully reply in round 14 ['NODE_5eaa634a-8589-4251-9779-f4435535cb17', 'NODE_523186bc-f4af-4044-a946-60003cdef368']

2026-04-15 17:14:19,061 fedbiomed INFO - breakpoint number 14 saved at /home/lchambon/Documents/FedBioMed/dev_tests/tests_local_parameters/fbm-components/fbm-researcher/var/experiments/Experiment_0000/breakpoint_0014

15

#### Check model parameters

In [5]:
print("\nList the training rounds : ", exp.aggregated_params().keys())

print("\nAccess the federated params for the last training round :")
print("\t- parameter data: ", exp.aggregated_params()[rounds - 1]['params'].keys())


List the training rounds :  dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14])

Access the federated params for the last training round :
	- parameter data:  dict_keys(['conv1.bias', 'conv2.weight', 'conv2.bias', 'fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias'])


#### Breakpoints

In [6]:
new_exp = Experiment.load_breakpoint('../../fbm-researcher/var/experiments/Experiment_0000/breakpoint_0014')

2026-04-15 17:15:19,975 fedbiomed INFO - Experimentation reload from ../../fbm-researcher/var/experiments/Experiment_0000/breakpoint_0014 successful!

In [7]:
new_exp.run_once(increase=True)

2026-04-15 17:15:23,310 fedbiomed INFO - Sampled nodes in round 15 ['NODE_5eaa634a-8589-4251-9779-f4435535cb17', 'NODE_523186bc-f4af-4044-a946-60003cdef368']

2026-04-15 17:15:23,317 fedbiomed INFO - Sending request 
					 To: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-04-15 17:15:23,317 fedbiomed INFO - Sending request 
					 To: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-04-15 17:15:23,533 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Node Name: Default Node Alias 
					 Round 16 | Iteration: 1/20 (5%) | Samples: 5/100
 					 Loss: 0.210073 
					 ---------

2026-04-15 17:15:24,024 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Node Name: Default Node Alias 
					 Round 16 | Iteration: 1/20 (5%) | Samples: 5/100
 					 Loss: 0.663331 
					 ---------

...

2026-04-15 17:15:30,198 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Node Name: Default Node Alias 
					 Round 16 | Iteration: 20/20 (100%) | Samples: 100/100
 					 Loss: 0.050097 
					 ---------

2026-04-15 17:15:30,254 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Node Name: Default Node Alias 
					 Round 16 | Iteration: 20/20 (100%) | Samples: 100/100
 					 Loss: 0.153597 
					 ---------

2026-04-15 17:15:30,591 fedbiomed INFO - Nodes that successfully reply in round 15 ['NODE_5eaa634a-8589-4251-9779-f4435535cb17', 'NODE_523186bc-f4af-4044-a946-60003cdef368']

2026-04-15 17:15:30,687 fedbiomed INFO - breakpoint number 15 saved at /home/lchambon/Documents/FedBioMed/dev_tests/tests_local_parameters/fbm-components/fbm-researcher/var/experiments/Experiment_0000/breakpoint_0015

1

# Test 2: with secagg and DP

### Model arguments and training arguments

In [11]:
model_args = {}

training_args = {
    'loader_args': { 'batch_size': 5, }, 
    'optimizer_args': {
        "lr" : 1e-3
    },
     'dp_args': # DP Arguments for differential privacy
    {
        "type": "central", 
        "sigma": 0.1, 
        "clip": 0.5
    },
    'num_updates': 20,
    'dry_run': False
}

### Create and run the experiment

In [12]:
from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.aggregators.fedavg import FedAverage
from fedbiomed.common.optimizers.optimizer import Optimizer
from fedbiomed.common.optimizers.declearn import ScaffoldServerModule
from fedbiomed.common.optimizers.declearn import YogiModule as FedYogi

tags =  ['#MNIST', '#dataset']
rounds = 15

exp = Experiment(tags=tags,
                 model_args=model_args,
                 training_plan_class=MyTrainingPlan,
                 training_args=training_args,
                 round_limit=rounds,
                 aggregator=FedAverage(),
                 tensorboard=True,
                 secagg=True,
                 save_breakpoints=True,
                 node_selection_strategy=None)

2026-04-15 17:31:49,673 fedbiomed INFO - Updating training data. This action will update FederatedDataset, and the nodes that will participate to the experiment.

2026-04-15 17:31:49,739 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_5eaa634a-8589-4251-9779-f4435535cb17

2026-04-15 17:31:49,740 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_523186bc-f4af-4044-a946-60003cdef368

In [13]:
exp.run()

2026-04-15 17:31:55,021 fedbiomed INFO - Sampled nodes in round 0 ['NODE_5eaa634a-8589-4251-9779-f4435535cb17', 'NODE_523186bc-f4af-4044-a946-60003cdef368']

2026-04-15 17:31:55,039 fedbiomed INFO - Sending request 
					 To: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-04-15 17:31:55,042 fedbiomed INFO - Sending request 
					 To: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-04-15 17:31:56,136 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Node Name: Default Node Alias 
					 Round 1 | Iteration: 1/20 (5%) | Samples: 8/100
 					 Loss: 2.946982 
					 ---------

2026-04-15 17:31:56,213 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Node Name: Default Node Alias 
					 Round 1 | Iteration: 1/20 (5%) | Samples: 4/100
 					 Loss: 4.189320 
					 ---------

...

2026-04-15 17:35:41,242 fedbiomed INFO - Validation is completed.

2026-04-15 17:35:41,242 fedbiomed INFO - Aggregating encrypted parameters. This process may take some time dependingon model size.

2026-04-15 17:35:41,391 fedbiomed INFO - Aggregating 2 parameters from 2 nodes.

2026-04-15 17:35:48,896 fedbiomed INFO - breakpoint number 14 saved at /home/lchambon/Documents/FedBioMed/dev_tests/tests_local_parameters/fbm-components/fbm-researcher/var/experiments/Experiment_0002/breakpoint_0014

15

#### Check model parameters

In [14]:
print("\nList the training rounds : ", exp.aggregated_params().keys())

print("\nAccess the federated params for the last training round :")
print("\t- parameter data: ", exp.aggregated_params()[rounds - 1]['params'].keys())


List the training rounds :  dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14])

Access the federated params for the last training round :
	- parameter data:  dict_keys(['conv1.bias', 'conv2.weight', 'conv2.bias', 'fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias'])


# Test 3: with Scaffold

## Define an experiment model and parameters

Declare a torch training plan MyTrainingPlan class to send for training on the node

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torchvision import  transforms

from fedbiomed.common.training_plans import TorchTrainingPlan
from fedbiomed.common.datamanager import DataManager
from fedbiomed.common.optimizers.optimizer import Optimizer
from fedbiomed.common.optimizers.declearn import ScaffoldClientModule
from fedbiomed.common.dataset import MnistDataset


# Here we define the training plan to be used.
# You can use any class name (here 'MyTrainingPlan')
class MyTrainingPlan2(TorchTrainingPlan):

    def init_dependencies(self):
        return ["from torchvision import transforms",
                "from fedbiomed.common.dataset import MnistDataset",
                "from fedbiomed.common.optimizers.optimizer import Optimizer",
                "from fedbiomed.common.optimizers.declearn import ScaffoldClientModule",
                "from fedbiomed.common.optimizers.declearn import AdamModule",
                "from fedbiomed.common.optimizers.declearn import RidgeRegularizer",
                "from torch.optim import Adam"]
        
    class Net(nn.Module):
        def __init__(self, model_args):
            super().__init__()
            self.conv1 = nn.Conv2d(1, 32, 3, 1)
            self.conv2 = nn.Conv2d(32, 64, 3, 1)
            self.dropout1 = nn.Dropout(0.25)
            self.dropout2 = nn.Dropout(0.5)
            self.fc1 = nn.Linear(9216, 128)
            self.fc2 = nn.Linear(128, 10)

        def forward(self, x):
            x = self.conv1(x)
            x = F.relu(x)
            x = self.conv2(x)
            x = F.relu(x)
            x = F.max_pool2d(x, 2)
            x = self.dropout1(x)
            x = torch.flatten(x, 1)
            x = self.fc1(x)
            x = F.relu(x)
            x = self.dropout2(x)
            x = self.fc2(x)
            output = F.log_softmax(x, dim=1)
            return output

    def init_model(self, model_args):
        model = self.Net(model_args = model_args)
        return model
        
    def init_optimizer(self, optimizer_args):
       # Defines and return a declearn optimizer
        return Optimizer(lr=optimizer_args['lr'], modules=[ScaffoldClientModule()])

    def training_data(self):
        transform = transforms.Normalize((0.1307,), (0.3081,))
        loader_arguments = { 'shuffle': True}
        
        dataset = MnistDataset(transform=transform)
        return DataManager(dataset, **loader_arguments)

    def training_step(self, data, target):
        output = self.model().forward(data)
        loss   = torch.nn.functional.nll_loss(output, target)
        return loss

    def tag_parameters(self, name):
        if name == 'conv1.weight':
            return {'local', 'persistent'}
        if name == 'conv2.bias':
            return {'persistent'}
        return set()
            

### Model arguments and training arguments

In [16]:
model_args = {}

training_args = {
    'loader_args': { 'batch_size': 5, }, 
    'optimizer_args': {
        "lr" : 1e-3
    },
    'num_updates': 20,
    'dry_run': False
}

### Create and run the experiment

In [17]:
from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.aggregators.fedavg import FedAverage
from fedbiomed.common.optimizers.optimizer import Optimizer
from fedbiomed.common.optimizers.declearn import ScaffoldServerModule
from fedbiomed.common.optimizers.declearn import YogiModule as FedYogi

tags =  ['#MNIST', '#dataset']
rounds = 15

exp = Experiment(tags=tags,
                 model_args=model_args,
                 training_plan_class=MyTrainingPlan2,
                 training_args=training_args,
                 round_limit=rounds,
                 aggregator=FedAverage(),
                 tensorboard=True,
                 save_breakpoints=True,
                 node_selection_strategy=None)

fed_opt = Optimizer(lr=.8, modules=[ScaffoldServerModule()])
exp.set_agg_optimizer(fed_opt)

2026-04-15 17:39:55,122 fedbiomed INFO - Updating training data. This action will update FederatedDataset, and the nodes that will participate to the experiment.

2026-04-15 17:39:55,180 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_5eaa634a-8589-4251-9779-f4435535cb17

2026-04-15 17:39:55,181 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_523186bc-f4af-4044-a946-60003cdef368

2026-04-15 17:39:55,191 fedbiomed WARNING - Using Declearn optimizers for aggregation and parameters tagged as local may lead to unexpected behaviours.

In [18]:
exp.run()

2026-04-15 17:40:00,381 fedbiomed INFO - Sampled nodes in round 0 ['NODE_5eaa634a-8589-4251-9779-f4435535cb17', 'NODE_523186bc-f4af-4044-a946-60003cdef368']

2026-04-15 17:40:00,388 fedbiomed INFO - Sending request 
					 To: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-04-15 17:40:00,389 fedbiomed INFO - Sending request 
					 To: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-04-15 17:40:00,818 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Node Name: Default Node Alias 
					 Round 1 | Iteration: 1/20 (5%) | Samples: 5/100
 					 Loss: 2.271706 
					 ---------

2026-04-15 17:40:01,168 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Node Name: Default Node Alias 
					 Round 1 | Iteration: 1/20 (5%) | Samples: 5/100
 					 Loss: 2.386757 
					 ---------

...

2026-04-15 17:41:47,822 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Node Name: Default Node Alias 
					 Round 15 | Iteration: 20/20 (100%) | Samples: 100/100
 					 Loss: 2.196840 
					 ---------

2026-04-15 17:41:48,011 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Node Name: Default Node Alias 
					 Round 15 | Iteration: 20/20 (100%) | Samples: 100/100
 					 Loss: 1.756389 
					 ---------

2026-04-15 17:41:48,317 fedbiomed INFO - Nodes that successfully reply in round 14 ['NODE_5eaa634a-8589-4251-9779-f4435535cb17', 'NODE_523186bc-f4af-4044-a946-60003cdef368']

2026-04-15 17:41:48,425 fedbiomed INFO - breakpoint number 14 saved at /home/lchambon/Documents/FedBioMed/dev_tests/tests_local_parameters/fbm-components/fbm-researcher/var/experiments/Experiment_0003/breakpoint_0014

15

#### Check model parameters

In [19]:
print("\nList the training rounds : ", exp.aggregated_params().keys())

print("\nAccess the federated params for the last training round :")
print("\t- parameter data: ", exp.aggregated_params()[rounds - 1]['params'].keys())


List the training rounds :  dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14])

Access the federated params for the last training round :
	- parameter data:  dict_keys(['conv1.bias', 'conv2.weight', 'conv2.bias', 'fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias'])


# Test 4: with Declearn optimizers and secagg

## Define an experiment model and parameters

Declare a torch training plan MyTrainingPlan class to send for training on the node

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torchvision import  transforms

from fedbiomed.common.training_plans import TorchTrainingPlan
from fedbiomed.common.datamanager import DataManager
from fedbiomed.common.optimizers.optimizer import Optimizer
from fedbiomed.common.optimizers.declearn import ScaffoldClientModule
from fedbiomed.common.dataset import MnistDataset


# Here we define the training plan to be used.
# You can use any class name (here 'MyTrainingPlan')
class MyTrainingPlan3(TorchTrainingPlan):

    def init_dependencies(self):
        return ["from torchvision import transforms",
                "from fedbiomed.common.dataset import MnistDataset",
                "from fedbiomed.common.optimizers.optimizer import Optimizer",
                "from fedbiomed.common.optimizers.declearn import ScaffoldClientModule",
                "from fedbiomed.common.optimizers.declearn import AdamModule",
                "from fedbiomed.common.optimizers.declearn import RidgeRegularizer",
                "from torch.optim import Adam"]
        
    class Net(nn.Module):
        def __init__(self, model_args):
            super().__init__()
            self.conv1 = nn.Conv2d(1, 32, 3, 1)
            self.conv2 = nn.Conv2d(32, 64, 3, 1)
            self.dropout1 = nn.Dropout(0.25)
            self.dropout2 = nn.Dropout(0.5)
            self.fc1 = nn.Linear(9216, 128)
            self.fc2 = nn.Linear(128, 10)

        def forward(self, x):
            x = self.conv1(x)
            x = F.relu(x)
            x = self.conv2(x)
            x = F.relu(x)
            x = F.max_pool2d(x, 2)
            x = self.dropout1(x)
            x = torch.flatten(x, 1)
            x = self.fc1(x)
            x = F.relu(x)
            x = self.dropout2(x)
            x = self.fc2(x)
            output = F.log_softmax(x, dim=1)
            return output

    def init_model(self, model_args):
        model = self.Net(model_args = model_args)
        return model

    def init_optimizer(self, optimizer_args):
        # Defines and return a declearn optimizer
        return Optimizer(lr=optimizer_args['lr'], modules=[AdamModule()], regularizers=[RidgeRegularizer()])

    def training_data(self):
        transform = transforms.Normalize((0.1307,), (0.3081,))
        loader_arguments = { 'shuffle': True}
        
        dataset = MnistDataset(transform=transform)
        return DataManager(dataset, **loader_arguments)

    def training_step(self, data, target):
        output = self.model().forward(data)
        loss   = torch.nn.functional.nll_loss(output, target)
        return loss

    def tag_parameters(self, name):
        if name == 'conv1.weight':
            return {'local', 'persistent'}
        if name == 'conv2.bias':
            return {'persistent'}
        return set()
            

### Model arguments and training arguments

In [21]:
model_args = {}

training_args = {
    'loader_args': { 'batch_size': 5, }, 
    'optimizer_args': {
        "lr" : 1e-3
    },
    'num_updates': 20,
    'dry_run': False
}

### Create and run the experiment

In [22]:
from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.aggregators.fedavg import FedAverage
from fedbiomed.common.optimizers.optimizer import Optimizer
from fedbiomed.common.optimizers.declearn import ScaffoldServerModule
from fedbiomed.common.optimizers.declearn import YogiModule as FedYogi

tags =  ['#MNIST', '#dataset']
rounds = 15

exp = Experiment(tags=tags,
                 model_args=model_args,
                 training_plan_class=MyTrainingPlan3,
                 training_args=training_args,
                 round_limit=rounds,
                 aggregator=FedAverage(),
                 tensorboard=True,
                 secagg=True,
                 save_breakpoints=True,
                 node_selection_strategy=None)

fed_opt = Optimizer(lr=.8, modules=[FedYogi()])
exp.set_agg_optimizer(fed_opt)

2026-04-15 17:45:25,081 fedbiomed INFO - Updating training data. This action will update FederatedDataset, and the nodes that will participate to the experiment.

2026-04-15 17:45:25,144 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_5eaa634a-8589-4251-9779-f4435535cb17

2026-04-15 17:45:25,145 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_523186bc-f4af-4044-a946-60003cdef368

2026-04-15 17:45:25,154 fedbiomed WARNING - Using Declearn optimizers for aggregation and parameters tagged as local may lead to unexpected behaviours.

In [23]:
exp.run()

2026-04-15 17:45:30,888 fedbiomed INFO - Sampled nodes in round 0 ['NODE_5eaa634a-8589-4251-9779-f4435535cb17', 'NODE_523186bc-f4af-4044-a946-60003cdef368']

2026-04-15 17:45:30,899 fedbiomed INFO - Sending request 
					 To: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-04-15 17:45:30,900 fedbiomed INFO - Sending request 
					 To: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-04-15 17:45:31,682 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Node Name: Default Node Alias 
					 Round 1 | Iteration: 1/20 (5%) | Samples: 5/100
 					 Loss: 2.241786 
					 ---------

2026-04-15 17:45:31,816 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Node Name: Default Node Alias 
					 Round 1 | Iteration: 1/20 (5%) | Samples: 5/100
 					 Loss: 2.292182 
					 ---------

...

2026-04-15 17:48:52,071 fedbiomed INFO - Validation is completed.

2026-04-15 17:48:52,072 fedbiomed INFO - Aggregating encrypted parameters. This process may take some time dependingon model size.

2026-04-15 17:48:52,221 fedbiomed INFO - Aggregating 2 parameters from 2 nodes.

2026-04-15 17:48:59,411 fedbiomed INFO - breakpoint number 14 saved at /home/lchambon/Documents/FedBioMed/dev_tests/tests_local_parameters/fbm-components/fbm-researcher/var/experiments/Experiment_0004/breakpoint_0014

15

#### Check model parameters

In [24]:
print("\nList the training rounds : ", exp.aggregated_params().keys())

print("\nAccess the federated params for the last training round :")
print("\t- parameter data: ", exp.aggregated_params()[rounds - 1]['params'].keys())


List the training rounds :  dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14])

Access the federated params for the last training round :
	- parameter data:  dict_keys(['conv1.bias', 'conv2.weight', 'conv2.bias', 'fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias'])
